# ALQAC 2026 — Kaggle Public Candidate

This notebook clones the private `TuanAnh` branch, runs the hybrid candidate on Public Test through the private-like inference boundary, and creates `submission.json`. It starts with the required two-case candidate smoke test and supports a separate full 50-case run after that gate passes. It never uploads to the leaderboard.

## 0. Kaggle prerequisites

Before Run All:

1. Set Notebook Settings → Accelerator → GPU T4.
2. Set Notebook Settings → Internet → On.
3. Add a `GITHUB_TOKEN` Secret with read access to the private repository.
4. Add the organizer-issued `ALQAC_TEAM_TOKEN` Secret.
5. Grant this notebook access to both secrets.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile

IS_KAGGLE = Path('/kaggle').exists()
assert IS_KAGGLE, 'This notebook is prepared for a Kaggle runtime'

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret('GITHUB_TOKEN')
raw_alqac_token = secrets.get_secret('ALQAC_TEAM_TOKEN')
alqac_token = raw_alqac_token.strip() if raw_alqac_token else ''
assert github_token, 'Missing Kaggle Secret: GITHUB_TOKEN'
assert alqac_token, 'Missing Kaggle Secret: ALQAC_TEAM_TOKEN'
os.environ['ALQAC_TEAM_TOKEN'] = alqac_token
print({'secrets_loaded': True})

## 1. Clone or update the exact private branch

Authentication uses a temporary `GIT_ASKPASS`; the token is never placed in the URL, source, or output.

In [ ]:
REPO_URL = 'https://github.com/NGBao1608/DL_K23_ALQAC2026.git'
BRANCH = 'TuanAnh'
PROJECT_ROOT = Path('/kaggle/working/DL_K23_ALQAC2026')

with tempfile.TemporaryDirectory() as temporary_dir:
    askpass = Path(temporary_dir) / 'git-askpass.sh'
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({
        'GITHUB_TOKEN': github_token,
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
    })
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)],
            env=git_env,
            check=True,
        )
    else:
        subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT_ROOT, env=git_env, check=True)
        subprocess.run(['git', 'checkout', BRANCH], cwd=PROJECT_ROOT, check=True)
        subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=PROJECT_ROOT, check=True)

current_branch = subprocess.check_output(
    ['git', 'branch', '--show-current'], cwd=PROJECT_ROOT, text=True
).strip()
current_commit = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_ROOT, text=True
).strip()
assert current_branch == BRANCH, f'Expected {BRANCH}, got {current_branch}'
assert (PROJECT_ROOT / 'pyproject.toml').exists()
assert (PROJECT_ROOT / 'data/raw/ALQAC2026_public_test.json').exists()
assert (PROJECT_ROOT / 'data/raw/corpus_law_pub.json').exists()
print({'branch': current_branch, 'commit': current_commit, 'root': str(PROJECT_ROOT)})

## 2. Install the package and verify the GPU

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)],
    check=True,
)

import torch

assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings'
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__})

## 3. Select the experiment and approved API budget

This notebook is configured for the hybrid candidate workflow. Start with `RUN_MODE='smoke'` to run exactly two cases. A cold-cache smoke needs an explicitly approved budget of `4` attempts (2 cases × 2 deterministic queries); a smoke using a complete cache can use `0`. After the candidate smoke passes on Kaggle T4, populate the complete 50-case cache with the approved baseline workflow if it is not already available. Then set `RUN_MODE='full'`, attach that complete cache through `CACHE_SEED_PATH`, and set `APPROVED_MAX_NETWORK_CALLS=0`. The full preflight must report `cache_misses=0`. Smoke and full use separate run directories automatically. Progress appears as one `ALQAC_PROGRESS` JSON line per case stage.

In [ ]:
EXPERIMENT = 'candidate'  # baseline | candidate
RUN_MODE = 'smoke'        # change to full only after GPU/API smoke PASS
APPROVED_MAX_NETWORK_CALLS = 4  # cold-cache smoke; use 0 only with a complete cache
CACHE_SEED_PATH = None  # optional for smoke; required for a fresh full-candidate session

assert EXPERIMENT in {'baseline', 'candidate'}
assert RUN_MODE in {'smoke', 'full'}
CONFIG_PATH = PROJECT_ROOT / f'configs/{EXPERIMENT}.yaml'
RUN_DIR = Path('/kaggle/working/alqac2026/outputs') / f'public_{EXPERIMENT}_{RUN_MODE}_{current_commit}'
CACHE_DB = Path('/kaggle/working/alqac2026/cache/case_api.sqlite')
LIMIT = 2 if RUN_MODE == 'smoke' else None
assert isinstance(APPROVED_MAX_NETWORK_CALLS, int) and APPROVED_MAX_NETWORK_CALLS >= 0
CACHE_DB.parent.mkdir(parents=True, exist_ok=True)
if CACHE_SEED_PATH is not None:
    seed = Path(CACHE_SEED_PATH)
    assert seed.is_file(), f'Cache seed does not exist: {seed}'
    if not CACHE_DB.exists():
        shutil.copy2(seed, CACHE_DB)
        print({'cache_seed_imported': str(seed)})
    else:
        print({'cache_seed_skipped': 'working cache already exists'})

print({
    'experiment': EXPERIMENT,
    'mode': RUN_MODE,
    'limit': LIMIT,
    'config': str(CONFIG_PATH),
    'run_dir': str(RUN_DIR),
    'cache_db': str(CACHE_DB),
    'approved_max_network_calls': APPROVED_MAX_NETWORK_CALLS,
})

## 4. Preflight and run the pipeline

The preflight is read-only with respect to the official API. It must fit inside the explicit budget before the pipeline starts. If a Kaggle session stops, restore the exported SQLite cache and resume only with the same experiment identity. Never use a smoke run directory for a full run.

In [ ]:
API_PLAN_PATH = Path('/kaggle/working/alqac2026') / f'api_plan_{EXPERIMENT}_{RUN_MODE}_{current_commit}.json'
plan_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts/plan_api_calls.py'),
    '--config', str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
    '--input', 'data/raw/ALQAC2026_public_test.json',
    '--cache-db', str(CACHE_DB),
    '--output', str(API_PLAN_PATH),
    '--approved-max-network-calls', str(APPROVED_MAX_NETWORK_CALLS),
]
if LIMIT is not None:
    plan_command.extend(['--limit', str(LIMIT)])
subprocess.run(plan_command, cwd=PROJECT_ROOT, check=True)
api_plan = json.loads(API_PLAN_PATH.read_text(encoding='utf-8'))
print('API PREFLIGHT:', {key: api_plan[key] for key in (
    'logical_queries', 'cache_hits', 'cache_misses',
    'approved_max_network_calls', 'known_local_cumulative_attempts'
)})
if api_plan['cache_misses'] > APPROVED_MAX_NETWORK_CALLS:
    raise RuntimeError(
        'API preflight blocked the run before any network request: '
        f"cache_misses={api_plan['cache_misses']} exceeds "
        f'APPROVED_MAX_NETWORK_CALLS={APPROVED_MAX_NETWORK_CALLS}. '
        'Attach the correct cache seed, or explicitly approve a sufficient '
        'budget before rerunning. Full candidate requires cache_misses=0 and budget=0.'
    )
if EXPERIMENT == 'candidate' and RUN_MODE == 'full':
    assert api_plan['cache_misses'] == 0, (
        'Full candidate must use a complete Case Content API cache', api_plan
    )
    assert APPROVED_MAX_NETWORK_CALLS == 0, (
        'Full candidate must not make new Case Content API calls', api_plan
    )

command = [
    sys.executable, '-u',
    str(PROJECT_ROOT / 'scripts/run_public.py'),
    '--config', str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
    '--input', 'data/raw/ALQAC2026_public_test.json',
    '--resume-run', str(RUN_DIR),
    '--cache-db', str(CACHE_DB),
    '--max-network-calls', str(APPROVED_MAX_NETWORK_CALLS),
]
if LIMIT is not None:
    command.extend(['--limit', str(LIMIT)])

process_env = os.environ.copy()
process_env['PYTHONUNBUFFERED'] = '1'
print('Streaming structured progress lines prefixed with ALQAC_PROGRESS...')
process = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    env=process_env,
    check=False,
)
if process.returncode != 0:
    diagnostics = {'returncode': process.returncode, 'run_dir': str(RUN_DIR)}
    for filename in ('manifest.json', 'api_plan.json', 'api_stats.json'):
        artifact = RUN_DIR / filename
        if artifact.exists():
            diagnostics[filename] = json.loads(artifact.read_text(encoding='utf-8'))
    failure_cache_export = Path('/kaggle/working/export/alqac_case_api_failed_run.sqlite')
    failure_cache_export.parent.mkdir(parents=True, exist_ok=True)
    if CACHE_DB.exists():
        shutil.copy2(CACHE_DB, failure_cache_export)
        diagnostics['preserve_cache_separately'] = str(failure_cache_export)
    print('PIPELINE FAILED — SAFE DIAGNOSTICS:')
    print(json.dumps(diagnostics, ensure_ascii=False, indent=2))
    root_error = diagnostics.get('manifest.json', {}).get('run', {}).get(
        'error', 'CLI failed before writing manifest.json'
    )
    raise RuntimeError(f'ALQAC pipeline failed: {root_error}')
print('Pipeline completed successfully.')

## 5. Validate and inspect metrics

In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

manifest = read_json(RUN_DIR / 'manifest.json')
validation = read_json(RUN_DIR / 'validation.json')
metrics = read_json(RUN_DIR / 'metrics.json')
api_stats = read_json(RUN_DIR / 'api_stats.json')
predictions = read_json(RUN_DIR / 'predictions.json')
expected_cases = 2 if RUN_MODE == 'smoke' else 50

assert manifest['run']['status'] == 'completed', manifest['run']
assert manifest['run']['completed'] == expected_cases, manifest['run']
assert validation['status'] == 'PASS', validation
assert validation['cases'] == expected_cases, validation
assert api_stats['run_network_attempts'] <= APPROVED_MAX_NETWORK_CALLS, api_stats

print('PER-CASE SUMMARY:')
for index, item in enumerate(predictions, start=1):
    print(json.dumps({
        'index': index,
        'total': len(predictions),
        'case_id': item['case_id'],
        'status': item['status'],
        'prediction': item['prediction'],
        'case_evidence_count': len(item['case_evidence']),
        'law_evidence_count': len(item['law_evidence']),
        'api_calls': item['api_calls'],
        'latency_seconds': round(item['latency_seconds'], 3),
        'error_type': item['error'].split(':', 1)[0] if item.get('error') else None,
    }, ensure_ascii=False, sort_keys=True))
print('VALIDATION:', validation)
print('METRICS:')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('API STATS:')
print(json.dumps(api_stats, ensure_ascii=False, indent=2))

## 6. Export submission artifacts and cache separately

Never upload a smoke file. The ZIP contains submission evidence only. The SQLite cache is exported separately and must never be uploaded as the leaderboard submission. Preserve it as a private cache seed for later Kaggle sessions.

In [ ]:
EXPORT_DIR = Path('/kaggle/working/export') / f'public_{EXPERIMENT}_{RUN_MODE}'
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

for filename in ('submission.json', 'validation.json', 'metrics.json', 'manifest.json', 'api_plan.json', 'api_stats.json'):
    shutil.copy2(RUN_DIR / filename, EXPORT_DIR / filename)

archive_path = shutil.make_archive(str(EXPORT_DIR), 'zip', root_dir=EXPORT_DIR)
CACHE_EXPORT_PATH = Path('/kaggle/working/export/alqac_case_api.sqlite')
shutil.copy2(CACHE_DB, CACHE_EXPORT_PATH)
print('Submission:', RUN_DIR / 'submission.json')
print('Download artifact archive:', archive_path)
print('Preserve cache separately:', CACHE_EXPORT_PATH)